In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["ACCELERATE_USE_CPU"] = "true"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "0"

In [2]:
import json
import re
import inspect
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

print("torch version:", torch.__version__)
print("mps available:", torch.backends.mps.is_available())
print("GRPOConfig signature:", inspect.signature(GRPOConfig))

torch version: 2.10.0
mps available: True
GRPOConfig signature: (output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 1e-06, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.

In [3]:
import torch
print("mps available:", torch.backends.mps.is_available())
print("default device test:", torch.device("cpu"))

mps available: True
default device test: cpu


In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["ACCELERATE_USE_CPU"] = "true"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "0"

import re
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

model_name = "Qwen/Qwen2.5-Math-1.5B"

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

TRAIN_PATH = "data/train_easy_math_120.jsonl"
VAL_PATH = "data/val_easy_math_30.jsonl"

train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)

print("train:", len(train_rows))
print("val:", len(val_rows))
print(train_rows[0])

def make_prompt(question: str) -> str:
    return f"""You are a math assistant.
Return only the final numeric answer.
Do not explain.

Examples:
Question: 12 * 13
Answer: 156

Question: 25 + 17
Answer: 42

Question: 84 / 6 + 15
Answer: 29

Now answer:
Question: {question}
Answer:"""

def extract_final_answer(text: str) -> str:
    text = text.strip()
    if not text:
        return ""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""
    first_line = lines[0]
    m = re.fullmatch(r"-?\d+", first_line)
    return m.group(0) if m else ""

def reward_format_and_correctness(completions, answer, **kwargs):
    rewards = []
    for completion, gold in zip(completions, answer):
        if isinstance(completion, list):
            text = completion[0]["content"].strip()
        else:
            text = str(completion).strip()

        pred = extract_final_answer(text)
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        first_line = lines[0] if lines else ""

        if pred == gold and first_line == gold:
            rewards.append(1.0)
        elif pred == gold:
            rewards.append(0.5)
        elif pred != "" and first_line == pred:
            rewards.append(0.1)
        else:
            rewards.append(-0.2)
    return rewards

def to_rl_record(row):
    return {
        "id": row.get("id", ""),
        "type": row.get("type", ""),
        "topic": row.get("topic", ""),
        "prompt": [{"role": "user", "content": make_prompt(row["question"])}],
        "answer": row["answer"],
        "question": row["question"],
    }

train_dataset = Dataset.from_list([to_rl_record(r) for r in train_rows])
val_dataset = Dataset.from_list([to_rl_record(r) for r in val_rows])

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
)
model.to("cpu")

print("model device:", next(model.parameters()).device)

peft_config = LoraConfig(
    r=2,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

grpo_config = GRPOConfig(
    output_dir="grpo_lora_qwen15b_math_cpu_out",
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    num_generations=2,
    generation_batch_size=2,
    max_completion_length=12,
    logging_steps=1,
    save_steps=20,
    num_train_epochs=1,
    report_to=[],
    use_cpu=True,
    dataloader_pin_memory=False,
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_dataset,
    reward_funcs=[reward_format_and_correctness],
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("trainer model device:", next(trainer.model.parameters()).device)

trainer.train()

train: 120
val: 30
{'id': 'train_001', 'type': 'train', 'topic': 'shopping_counting', 'question': 'Maya bought 11 apples and 5 oranges. She used 6 apples to bake a pie and then bought 4 more oranges. How many pieces of fruit does she have now?', 'answer': '14'}


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

model device: cpu


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainer model device: cpu


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


TrainOutput(global_step=120, training_loss=0.0, metrics={'train_runtime': 34056.2019, 'train_samples_per_second': 0.004, 'train_steps_per_second': 0.004, 'total_flos': 0.0, 'train_loss': 0.0})

In [6]:
print(inspect.signature(GRPOConfig))

(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 1e-06, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_com

In [7]:
train_result = trainer.train()

training_time_sec = train_result.metrics.get("train_runtime")
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in trainer.model.parameters())
trainable_percent = 100 * trainable_params / total_params

print("training_time_sec:", training_time_sec)
print("trainable_params:", trainable_params)
print("total_params:", total_params)
print("trainable_percent:", trainable_percent)

Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


training_time_sec: 30838.7524
trainable_params: 272384
total_params: 1543986688
trainable_percent: 0.01764160287889736


In [11]:
import os

print("Output dir exists:", os.path.exists("grpo_lora_qwen15b_math_cpu_out"))
print("Files:", os.listdir("grpo_lora_qwen15b_math_cpu_out"))

Output dir exists: True
Files: ['checkpoint-80', 'checkpoint-20', 'checkpoint-100', 'checkpoint-60', 'README.md', 'checkpoint-40', 'checkpoint-120']


In [12]:
from peft import PeftModel
# -------------------------
# Load RL adapter for eval
# -------------------------
rl_adapter_path = "grpo_lora_qwen15b_math_cpu_out/checkpoint-120"   # change if checkpoint folder is needed

base_model = AutoModelForCausalLM.from_pretrained(model_name)
rl_model = PeftModel.from_pretrained(base_model, rl_adapter_path)
rl_model.eval()

def run_model_rl(question: str) -> str:
    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = rl_model.generate(
        **inputs,
        max_new_tokens=12,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # strip prompt if echoed
    generated_text = full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text
    return generated_text.strip()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [17]:
# -------------------------
# Evaluate one dataset
# -------------------------
def evaluate_dataset(dataset, run_fn):
    rows = []

    for ex in dataset:
        raw_output = run_fn(ex["question"])
        pred = extract_final_answer(raw_output)

        rows.append({
            "id": ex["id"],
            "question": ex["question"],
            "gold": ex["answer"],
            "raw_output": raw_output,
            "pred": pred,
            "correct": pred == ex["answer"],
        })

    return pd.DataFrame(rows)

In [13]:
# -------------------------
# Multi-run evaluation
# -------------------------
def evaluate_dataset_multi_run(dataset, run_fn, n_runs=3):
    all_rows = []

    for run_idx in range(n_runs):
        df = evaluate_dataset(dataset, run_fn).copy()
        df = df.rename(columns={
            "raw_output": f"raw_output_run{run_idx+1}",
            "pred": f"pred_run{run_idx+1}",
            "correct": f"correct_run{run_idx+1}",
        })
        all_rows.append(df)

    merged = all_rows[0]
    for df in all_rows[1:]:
        merged = merged.merge(df, on=["id", "question", "gold"], how="inner")

    correct_cols = [f"correct_run{i+1}" for i in range(n_runs)]
    pred_cols = [f"pred_run{i+1}" for i in range(n_runs)]
    raw_cols = [f"raw_output_run{i+1}" for i in range(n_runs)]

    merged["majority_correct"] = merged[correct_cols].sum(axis=1) >= (n_runs // 2 + 1)
    merged["consistency"] = merged[pred_cols].nunique(axis=1) == 1
    merged["avg_token_length"] = merged[raw_cols].apply(
        lambda row: sum(len(str(x).split()) for x in row) / n_runs,
        axis=1
    )

    return merged

In [19]:
# Example:
import pandas as pd
benchmark_50 = load_jsonl("data/benchmark_50.jsonl")   # if this returns id/question/answer rows

df_rl = evaluate_dataset_multi_run(benchmark_50, run_model_rl, n_runs=3)
df_rl.head()

,id,question,gold,raw_output_run1,pred_run1,correct_run1,raw_output_run2,pred_run2,correct_run2,raw_output_run3,pred_run3,correct_run3,majority_correct,consistency,avg_token_length
0,1,17 * 24,408,408\n\nQuestion: 100 -,408,True,408\n\nQuestion: 100 -,408,True,408\n\nQuestion: 100 -,408,True,True,True,4.0
1,2,125 + 378 - 249,254,254\n\nQuestion: 12 *,254,True,254\n\nQuestion: 12 *,254,True,254\n\nQuestion: 12 *,254,True,True,True,4.0
2,3,36 / 4 + 7 * 3,30,33\n\nQuestion: 100 -,33,False,33\n\nQuestion: 100 -,33,False,33\n\nQuestion: 100 -,33,False,False,True,4.0
3,4,(18 + 7) * 4,100,100\n\nQuestion: 12 *,100,True,100\n\nQuestion: 12 *,100,True,100\n\nQuestion: 12 *,100,True,True,True,4.0
4,5,84 / 6 + 15,29,29\n\nQuestion: 12 * 1,29,True,29\n\nQuestion: 12 * 1,29,True,29\n\nQuestion: 12 * 1,29,True,True,True,5.0


In [ ]:
RL_METADATA = {
    "condition": "grpo_lora_r2_cpu",
    "training_time_sec": 30838.75,
    "trainable_params": 272384,
    "total_params": 1543986688,
    "trainable_percent": 0.017642,
}